In [16]:
import cv2
import numpy as np

In [17]:
# 1. Configuration Dictionary: Define your multiple targets
# Format: "Name": (lower_hsv, upper_hsv, UI_draw_color_BGR)
targets = {
    "green_arm": {
        "lower": np.array([35, 100, 100]),
        "upper": np.array([85, 255, 255]),
        "color": (0, 255, 0) # Green for UI
    },
    "blue_arm": {
        "lower": np.array([100, 150, 0]),
        "upper": np.array([140, 255, 255]),
        "color": (255, 0, 0) # Blue for UI
    },
    # Add as many targets here as you need
}

# 2. Dictionary to store individual trajectory lists for each target
trajectories = {name: [] for name in targets.keys()}

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret: break
    
    # Convert original frame to HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    # Loop through each target in the dictionary
    for name, params in targets.items():
        # Create a mask specifically for this target's color range
        mask = cv2.inRange(hsv, params["lower"], params["upper"])
        
        # Clean up noise
        mask = cv2.erode(mask, None, iterations=2)
        mask = cv2.dilate(mask, None, iterations=2)
        
        # Find contours for this specific mask
        contours, _ = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if len(contours) > 0:
            c = max(contours, key=cv2.contourArea)
            M = cv2.moments(c)
            
            if M["m00"] > 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                
                # Append coordinates to this specific target's trajectory array
                trajectories[name].append((cx, cy))
                
                # Draw visual markers on the original high-res frame using its assigned color
                cv2.circle(frame, (cx, cy), 10, params["color"], -1)
                cv2.drawContours(frame, [c], -1, params["color"], 3)
                
                # Add a text label so you know which is which in the preview
                cv2.putText(frame, name, (cx - 20, cy - 20), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, params["color"], 2)

    # --- DISPLAY FIX ---
    # Scale down a copy of the fully annotated frame for the preview window
    scale = 0.5  
    preview_width = int(frame.shape[1] * scale)
    preview_height = int(frame.shape[0] * scale)
    frame_preview = cv2.resize(frame, (preview_width, preview_height))
    
    # We only show the main tracking window to avoid cluttering your screen 
    # with multiple mask windows
    cv2.imshow("Multi-Tracking", frame_preview)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# Example of how you would access the data afterwards:
# print("Green arm total frames tracked:", len(trajectories["green_arm"]))


In [18]:
trajectories

{'green_arm': [(231, 468),
  (250, 424),
  (266, 391),
  (270, 374),
  (279, 361),
  (286, 359),
  (277, 355),
  (277, 355),
  (276, 359),
  (272, 358),
  (271, 353),
  (268, 348),
  (266, 343),
  (269, 343),
  (269, 338),
  (271, 338),
  (262, 327),
  (275, 340),
  (272, 331),
  (273, 330),
  (267, 320),
  (268, 318),
  (280, 328),
  (285, 327),
  (285, 320),
  (285, 311),
  (303, 324),
  (309, 317),
  (314, 308),
  (307, 308),
  (311, 294),
  (297, 303),
  (279, 324),
  (252, 344),
  (255, 385),
  (5, 305),
  (19, 299),
  (33, 294),
  (48, 290),
  (67, 288),
  (98, 283),
  (98, 283),
  (105, 281),
  (113, 279),
  (120, 276),
  (126, 273),
  (134, 270),
  (138, 267),
  (143, 265),
  (148, 263),
  (153, 262),
  (158, 261),
  (162, 261),
  (166, 260),
  (170, 260),
  (172, 260),
  (175, 260),
  (178, 260),
  (180, 260),
  (181, 260),
  (182, 260),
  (183, 260),
  (184, 260),
  (186, 261),
  (188, 261),
  (190, 261),
  (192, 260),
  (194, 259),
  (195, 258),
  (196, 257),
  (197, 256),
 